In [1]:
import pandas as pd

df = pd.read_hdf(
    "pems-bay.h5"
)

data = df.values

print(data.shape)

(52116, 325)


In [2]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

data_scaled = scaler.fit_transform(data)

print(data_scaled.shape)

(52116, 325)


In [3]:
import numpy as np

SEQ_LEN = 12

X = []
y = []

for i in range(len(data_scaled)-SEQ_LEN):

    X.append(
        data_scaled[i:i+SEQ_LEN]
    )

    y.append(
        data_scaled[i+SEQ_LEN]
    )


X = np.array(X)
y = np.array(y)


print(X.shape)
print(y.shape)

(52104, 12, 325)
(52104, 325)


In [4]:
train_size = int(len(X)*0.7)
val_size = int(len(X)*0.1)


X_train = X[:train_size]
y_train = y[:train_size]


X_val = X[
    train_size:
    train_size+val_size
]

y_val = y[
    train_size:
    train_size+val_size
]


X_test = X[
    train_size+val_size:
]

y_test = y[
    train_size+val_size:
]


print(X_train.shape)
print(X_test.shape)

(36472, 12, 325)
(10422, 12, 325)


In [5]:
import torch

X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)


X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

In [6]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader


train_dataset = TensorDataset(
    X_train,
    y_train
)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=False
)

In [7]:
import pickle

with open("adj_mx.pkl", "rb") as f:
    sensor_ids, sensor_id_to_ind, adj_mx = pickle.load(
        f,
        encoding="latin1"
    )

print(type(adj_mx))
print(adj_mx.shape)

<class 'numpy.ndarray'>
(207, 207)


In [8]:
print("Min:", adj_mx.min())
print("Max:", adj_mx.max())

print("Connections:")
print((adj_mx > 0).sum())

Min: 0.0
Max: 1.0
Connections:
1722


In [9]:
import numpy as np

A = adj_mx

# add self connections
A = A + np.eye(A.shape[0])


degree = np.sum(A, axis=1)

D_inv_sqrt = np.diag(
    np.power(degree, -0.5)
)

A_hat = (
    D_inv_sqrt
    @ A
    @ D_inv_sqrt
)

print(A_hat.shape)

(207, 207)


In [10]:
import torch

A_hat = torch.FloatTensor(A_hat)

In [11]:
import torch.nn as nn


class MultiScaleTemporalConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv3 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(3,1),
            padding=(1,0)
        )

        self.conv5 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(5,1),
            padding=(2,0)
        )

        self.conv7 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(7,1),
            padding=(3,0)
        )

    def forward(self,x):

        out3 = self.conv3(x)

        out5 = self.conv5(x)

        out7 = self.conv7(x)

        return torch.relu(
            out3 + out5 + out7
        )

In [12]:
class AdaptiveGraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        channels
    ):

        super().__init__()

        self.node_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                16
            )
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        adaptive_adj = torch.softmax(
            torch.relu(
                self.node_embeddings
                @
                self.node_embeddings.T
            ),
            dim=1
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            adaptive_adj,
            x
        )

        x = x.permute(
            0,
            2,
            3,
            1
        )

        x = self.weight(x)

        x = x.permute(
            0,
            3,
            1,
            2
        )

        return torch.relu(x)

In [13]:
class AdaptiveMultiScaleSTGCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.temp1 = MultiScaleTemporalConv(
            1,
            32
        )

        self.graph = AdaptiveGraphConv(
            num_nodes=325,
            channels=32
        )

        self.temp2 = MultiScaleTemporalConv(
            32,
            32
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self,x):

        x = x.unsqueeze(1)

        x = self.temp1(x)

        x = self.graph(x)

        x = self.temp2(x)

        x = x.mean(dim=2)

        x = x.permute(
            0,
            2,
            1
        )

        x = self.fc(x)

        x = x.squeeze(-1)

        return x

In [14]:
model = AdaptiveMultiScaleSTGCN()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 325])
torch.Size([64, 325])


In [15]:
model = AdaptiveMultiScaleSTGCN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [16]:
epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0


    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()


        predictions = model(
            X_batch
        )


        loss = criterion(
            predictions,
            y_batch
        )


        loss.backward()


        optimizer.step()


        total_loss += loss.item()


    avg_loss = total_loss / len(train_loader)


    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {avg_loss:.6f}"
    )

Epoch 1/30 Loss: 0.005765
Epoch 2/30 Loss: 0.000721
Epoch 3/30 Loss: 0.000601
Epoch 4/30 Loss: 0.000593
Epoch 5/30 Loss: 0.000582
Epoch 6/30 Loss: 0.000580
Epoch 7/30 Loss: 0.000574
Epoch 8/30 Loss: 0.000566
Epoch 9/30 Loss: 0.000559
Epoch 10/30 Loss: 0.000554
Epoch 11/30 Loss: 0.000548
Epoch 12/30 Loss: 0.000544
Epoch 13/30 Loss: 0.000539
Epoch 14/30 Loss: 0.000535
Epoch 15/30 Loss: 0.000532
Epoch 16/30 Loss: 0.000527
Epoch 17/30 Loss: 0.000525
Epoch 18/30 Loss: 0.000522
Epoch 19/30 Loss: 0.000520
Epoch 20/30 Loss: 0.000518
Epoch 21/30 Loss: 0.000516
Epoch 22/30 Loss: 0.000513
Epoch 23/30 Loss: 0.000511
Epoch 24/30 Loss: 0.000509
Epoch 25/30 Loss: 0.000507
Epoch 26/30 Loss: 0.000505
Epoch 27/30 Loss: 0.000504
Epoch 28/30 Loss: 0.000502
Epoch 29/30 Loss: 0.000500
Epoch 30/30 Loss: 0.000499


In [17]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np


test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.013674469
RMSE: 0.024159545
